# 평가 Gold Set 생성

## 목적

2만 개 stratified sample 도서 DB를 활용하여 retrieval/rerank 모델 실험용 평가셋을 생성한다.

현재 단계에서는 전체 사람 검수가 어렵기 때문에, LLM judge를 활용해 후보 도서에 대한 relevance pseudo-label을 생성하고, 이를 기반으로 1차 gold candidate set을 구성한다.

---

## 입력 데이터

 1. `scenario_data.json`

- 경로: `evaluation/dataset/scenario_data.json`
- 구성:
  - `onboarding_data`: 사용자 장기 선호 정보
  - `session_data`: 현재 대화 세션에서 추출된 사용자 요청 정보

 2. 도서 DB

- 약 2만 개 stratified sample 도서 데이터
- BM25 / Dense / Hybrid 검색 대상
- ISBN 기준 도서 메타데이터 조회 대상

---

## 생성 과정

1. `session_data`와 `onboarding_data`를 기반으로 검색 질의를 구성한다.

2. 도서 DB를 대상으로 BM25, Dense, Hybrid retriever를 각각 실행하여 Top-40 후보를 추출한다.

3. 세 retriever 결과를 merge하고, ISBN 기준으로 중복 제거한다.

4. 중복 제거된 ISBN을 기준으로 DB에서 도서 메타데이터를 조회한다.

5. LLM judge가 각 후보 도서의 relevance를 `0~3 grade`로 판단한다.

   - `session_data`를 우선 기준으로 사용한다.
   - `use_onboarding=true`인 경우 `onboarding_data`를 보조 기준으로 반영한다.
   - `session_data`와 `onboarding_data`가 충돌하면 `session_data`를 우선한다.
   - retrieval rank, score, source는 LLM judge 입력에서 제외한다.
   - 단, `retrieval_info`는 분석 및 hard negative 추출을 위해 출력 데이터에는 저장한다.

6. `relevance_grade`를 기반으로 label을 생성한다.

```python
binary_label = 1 if relevance_grade >= 2 else 0
```

출력 항목 예시:

- `relevance_grade`: 0, 1, 2, 3
- `binary_label`: 0 또는 1
- `retrieval_info`: source/rank/score 정보


| Grade | 의미              | 처리 기준                         |
| ----: | --------------- | ----------------------------- |
|     3 | 매우 적합           | `gold_isbns` 후보               |
|     2 | 적합하지만 일부 조건이 약함 | `gold_isbns` 후보 또는 검수 우선 후보   |
|     1 | 부분 관련, 애매함      | borderline / hard negative 후보 |
|     0 | 부적합             | negative                      |
7. 향후 10만 개 전체 데이터 확장 시에는 전체 사람 검수가 아니라 다음 후보를 중심으로 선별적 사람 검수를 적용한다.
- grade=1~2인 후보
- LLM confidence가 낮은 후보
- retrieval rank는 높지만 grade=0인 후보
- 여러 retriever에서 중복 등장했지만 낮은 grade를 받은 후보
- query별 positive 수가 부족한 case

In [15]:
import os

os.chdir("/home/ubuntu/work/dahyeon/backend")
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [16]:
import json
from app.modules.RAG.retriever import (
                                    full_bm25_with_onboarding,
                                    full_dense_with_onboarding,
                                    full_hybrid_with_onboarding,
                                    make_onboarding_result)
import warnings
warnings.filterwarnings('ignore')

In [17]:
sample_result = {
    "keyword_query": ["한국소설", "에세이", "도시생활", "힐링", "문학적"],
    "semantic_query": "도시 생활에 지쳤을 때 힐링이 되는 문학적인 한국 작가 소설·에세이",
    "filters": {
      "cate_depth1": ["소설", "시/에세이"],
      "custom_constraints": ["한국 작가 우선", "문학적인 문체"]
    },
    "constraints": {},
    "score_boost": { "cate_depth2": ["한국소설", "테마에세이"], "subject": ["도시생활", "힐링"] },
    "session_signals": {
      "mood": ["negative_stressed", "recovery_relax"],
      "purpose": "재미",
      "reading_level": "medium"
    },
    "onboarding_signals": { "length_soft": { "operator": "lte", "value": 250 } }
  }
    
# results = full_bm25_with_onboarding(sample_result, size=40)

# for r in results:
#     print(
#         r["rank"],
#         r.get("title"),
#         r.get("large_cate"),
#         r.get("score"),
#         r.get("main_score"),
#         r.get("onboarding_score"),
#         r.get("disliked_penalty"),
#         r.get("disliked_similarity")
    
re1_bm25 = full_bm25_with_onboarding(sample_result, size=40)
re1_dense = full_dense_with_onboarding(sample_result, size=40)
re1_hybrid = full_hybrid_with_onboarding(sample_result, size=40)

In [18]:

def deduplicate_results(results_list, key="isbn"):
    merged = []
    seen = set()

    for results in results_list:
        for item in results:

            item_key = item.get(key)

            if item_key in seen:
                continue

            seen.add(item_key)
            merged.append(item)

    return merged


# =========================
# full 결과 통합
# =========================

full_merged = deduplicate_results([
    re1_bm25,
    re1_dense,
    re1_hybrid
])

full_result_json = {
    "query": sample_result,
    "results": full_merged
}

#with open("full_results.json", "w", encoding="utf-8") as f:
#    json.dump(full_result_json, f, ensure_ascii=False, indent=2)

def simplify_item(item, source_name):
    return {
        "isbn": item.get("isbn"),
        "title": item.get("title"),
        "author": item.get("author"),
        "publisher": item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page": item.get("page"),

        "large_cate": item.get("large_cate"),
        "mid_cate": item.get("mid_cate"),
        "small_cate": item.get("small_cate"),

        "book_intro": item.get("book_intro"),
        "book_index": item.get("book_index"),

        "review_count": item.get("review_count"),
        "review_text": item.get("review_text"),

        # "retrieval_source": source_name,
        # "retrieval_rank": item.get("rank"),
        # "retrieval_score": item.get("score")
    }


def merge_with_sources(result_groups, key="isbn"):
    merged = {}

    for source_name, results in result_groups.items():
        for item in results:
            item_key = item.get(key)

            if item_key is None:
                continue

            simplified = simplify_item(item, source_name)

            if item_key not in merged:
                base = {
                    k: v for k, v in simplified.items()
                    if not k.startswith("retrieval_")
                }

                base["retrieval_info"] = []
                merged[item_key] = base

            merged[item_key]["retrieval_info"].append({
                "source": source_name,
                "rank": item.get("rank"),
                "score": item.get("score")
            })

    return list(merged.values())

full_books = merge_with_sources({
    "bm25": re1_bm25,
    "dense": re1_dense,
    "hybrid": re1_hybrid
})

#with open("full_books_for_labeling.json", "w", encoding="utf-8") as f:
#   json.dump(full_books, f, ensure_ascii=False, indent=2)

In [19]:
print(len(full_books))
full_books[0]

86


86


{'isbn': '9788973379552',
 'title': '화요일의 동물원 - 꿈을 찾는 이들에게 보내는 희망과 위안의 메시지',
 'author': '박민정 저, 사진',
 'publisher': '해냄',
 'publish_date': '2008-07-07',
 'page': 231,
 'large_cate': ['시/에세이'],
 'mid_cate': ['나라별 에세이'],
 'small_cate': ['한국에세이'],
 'book_intro': '4년 동안 160번의 화요일을 동물원에서 보낸 후 사랑과 행복, 그리고 영혼의 깨달음을 한 편 한 편의 우화로 써내려간 힐링 에세이.  동물들의 눈빛과 손짓을 통해 마음을 다독이는 영혼의 치유 에세이다. ‘사회라는 울타리와 동물원의 울타리, 그 속에서 울고 웃으며 사는 동물과 사람은 결국 비슷한 존재가 아닐까?’라는 상상력에서 출발한 이 책에는, 고집스럽게 동물들을 촬영하면서 깨달은 90가지 지혜가 담겨 있다.',
 'book_index': '',
 'review_count': 0,
 'review_text': '',
 'retrieval_info': [{'source': 'bm25', 'rank': 1, 'score': 0.787005596361985},
  {'source': 'hybrid', 'rank': 3, 'score': 0.4360548155890448}]}

In [ ]:
import json
import uuid
import time
import random
import requests
from tqdm import tqdm
from app.core.config import settings

URL = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"

CLOVA_API_KEY = settings.CLOVA_API_KEY


SYSTEM_PROMPT = """
    당신은 매우 보수적으로 책 추천 적합성을 판단하는 평가자입니다.

    입력으로는 다음 두 정보가 주어집니다.

    1. request: 사용자의 검색 요청 전체
    - keyword_query
    - semantic_query
    - filters
    - constraints
    - score_boost
    - session_signals
    - onboarding_signals

    2. book: 추천 후보 도서 정보

    [판정 기준]
    - 현재 사용자 요청의 핵심 의도와 조건을 우선적으로 판단하세요.
    - 사용자 요청의 모든 표현이 책 정보에 직접 등장할 필요는 없습니다.
    - 제목, 소개, 목차, 카테고리, 리뷰 등을 종합하여 의미적 적합성을 판단하세요.
    - 책이 제공하는 독서 경험, 분위기, 주제, 대상 독자가 사용자 요청과 충분히 일치하면 높은 점수를 부여하세요.
    - 일부 키워드만 우연히 일치하거나, 장르/주제/대상/분위기가 실제로 맞지 않으면 낮은 점수를 부여하세요.
    - filters, constraints는 명시 조건으로 보고 우선 고려하세요.
    - onboarding_signals와 session_signals는 사용자 선호 신호이므로 보조적으로 반영하세요.
    - 단, soft 조건은 약간 벗어나도 핵심 요청 적합성이 높으면 높은 점수를 부여할 수 있습니다.
    - 정보가 부족하여 핵심 조건 충족 여부를 판단할 수 없으면 낮은 점수를 부여하세요.
    - 요청과 직접 관련 없는 책이 단순히 키워드만 포함한 경우에는 낮은 점수를 부여하세요.

    출력 형식:
    {
      "relevance_grade": 0 또는 1 또는 2 또는 3
      "reason": "판정 근거를 간략히 설명하는 텍스트"
    }
    """


def make_headers():
    return {
        "Authorization": f"Bearer {CLOVA_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "text/event-stream"
    }


def parse_hcx_response(text):

    # =========================
    # 1. SSE 형식 처리
    # =========================
    if "event:" in text:

        last_data = None

        for block in text.split("\n\n"):

            lines = block.strip().splitlines()

            event_type = None
            data_text = None

            for line in lines:

                if line.startswith("event:"):
                    event_type = line.replace("event:", "").strip()

                elif line.startswith("data:"):
                    data_text = line.replace("data:", "").strip()

            if event_type == "result" and data_text:
                last_data = data_text

        if last_data is not None:
            data = json.loads(last_data)
            return data["message"]["content"]

    # =========================
    # 2. 일반 JSON 응답 처리
    # =========================
    try:
        data = json.loads(text)

        if "result" in data:
            return data["result"]["message"]["content"]

        if "message" in data:
            return data["message"]["content"]

    except:
        pass

    raise ValueError(f"응답 파싱 실패:\n{text[:1000]}")


def extract_grade(llm_text):
    try:
        obj = json.loads(llm_text)
        grade = obj.get("relevance_grade", obj.get("grade", obj.get("label")))
        if grade is None:
            return 0
        grade = int(grade)
        if grade not in [0, 1, 2, 3]:
            return 0
        return grade
    except Exception:
        # 파싱 실패하면 보수적으로 0
        return 0


def grade_to_binary(grade):
    return 1 if grade >= 2 else 0


def make_user_prompt(request_data, book):
    payload = {
        "request": request_data,
        "book": book
    }

    return json.dumps(payload, ensure_ascii=False, indent=2)


def label_one_book(request_data, book, max_retries=5):
    user_prompt = make_user_prompt(request_data, book)

    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        "topP": 0.1,
        "topK": 0,
        "max_tokens": 100,
        "temperature": 0.0,
        "repetitionPenalty": 1.0,
        "includeAiFilters": False,
        "seed": 42
    }

    for attempt in range(max_retries):
        try:
            res = requests.post(
                URL,
                headers=make_headers(),
                json=body,
                timeout=60
            )

            if res.status_code == 200:
                llm_text = parse_hcx_response(res.text)
                grade = extract_grade(llm_text)
                binary_label = grade_to_binary(grade)

                return {
                    **book,
                    "relevance_grade": grade,
                    "binary_label": binary_label,
                    "llm_raw_output": llm_text
                }

            if res.status_code in [429, 500, 502, 503, 504]:
                raise Exception(f"{res.status_code} / {res.text[:500]}")

            raise Exception(f"{res.status_code} / {res.text[:500]}")

        except Exception as e:
            if attempt == max_retries - 1:
                return {
                    **book,
                    "relevance_grade": 0,
                    "binary_label": 0,
                    "llm_error": str(e)
                }

            wait = min(30, (2 ** attempt) + random.uniform(0, 1))
            print(f"[재시도 {attempt+1}/{max_retries}] {wait:.1f}초 대기 → {e}")
            time.sleep(wait)

labeled_full_books = []

for book in tqdm(full_books):
    labeled = label_one_book(
        request_data=sample_result,
        book=book
    )

    labeled_full_books.append(labeled)

with open("full_books_labeled1.json", "w", encoding="utf-8") as f:
    json.dump(labeled_full_books, f, ensure_ascii=False, indent=2)


100%|██████████| 86/86 [17:45<00:00, 12.39s/it]


In [21]:
labeled_full_books[0]

{'isbn': '9788973379552',
 'title': '화요일의 동물원 - 꿈을 찾는 이들에게 보내는 희망과 위안의 메시지',
 'author': '박민정 저, 사진',
 'publisher': '해냄',
 'publish_date': '2008-07-07',
 'page': 231,
 'large_cate': ['시/에세이'],
 'mid_cate': ['나라별 에세이'],
 'small_cate': ['한국에세이'],
 'book_intro': '4년 동안 160번의 화요일을 동물원에서 보낸 후 사랑과 행복, 그리고 영혼의 깨달음을 한 편 한 편의 우화로 써내려간 힐링 에세이.  동물들의 눈빛과 손짓을 통해 마음을 다독이는 영혼의 치유 에세이다. ‘사회라는 울타리와 동물원의 울타리, 그 속에서 울고 웃으며 사는 동물과 사람은 결국 비슷한 존재가 아닐까?’라는 상상력에서 출발한 이 책에는, 고집스럽게 동물들을 촬영하면서 깨달은 90가지 지혜가 담겨 있다.',
 'book_index': '',
 'review_count': 0,
 'review_text': '',
 'retrieval_info': [{'source': 'bm25', 'rank': 1, 'score': 0.787005596361985},
  {'source': 'hybrid', 'rank': 3, 'score': 0.4360548155890448}],
 'relevance_grade': 0,
 'binary_label': 0,
 'llm_raw_output': '{\n  "relevance_grade": 2\n}\n\n**판정 이유:**  \n1. **카테고리/작가 조건 충족**  \n   - 시/에세이 카테고리, 한국 작가, 문학적인 문체(우화 형식) → 필터 조건 충족.\n\n2. **주제 부분 부분 일치**  \n   - "힐링" 키워드와 치유 메시지 강조 → 주제 일부 부합.  \n   - "도시생활"과의 직접적 연관성 약함 (동물원 경험 중심).  \n\n3. **길이/독서 

In [23]:
with open("/home/ubuntu/work/somin/evaluation/dataset/full_books_labeled1.json", "w", encoding="utf-8") as f:
    json.dump(labeled_full_books, f, ensure_ascii=False, indent=2)

In [ ]:
evaluation/dataset